# Phase 8 - Quick Pipeline Benchmark

This notebook runs the isolated stratified smoke-test pipeline. Quick-run metrics verify execution and performance only; they must not replace full-data research results.

In [1]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'configs' / 'config.yaml'
print(f'Project root: {project_root}')

Project root: e:\Paper Multiclass-Intrusion-Detection-System


In [2]:
# Cell 2 - Inspect the configured quick-run limits and isolated workspace.
from src.data_loading import load_config, resolve_project_path

config = load_config(config_path)
quick_config = config['pipeline']['quick_run']
quick_root = resolve_project_path(quick_config['root_dir'], project_root)

pd.Series(quick_config, name='configured_value').to_frame()

,configured_value
root_dir,data/interim/quick_run
train_rows,100000
test_rows,25000
minimum_train_rows_per_class,100
minimum_test_rows_per_class,20
autoencoder_epochs,5
lightgbm_estimators,50


In [3]:
# Cell 3 - Run or reuse the isolated all-scenario quick pipeline.
from run_pipeline import configure_logging
from src.quick_run import run_quick_pipeline

configure_logging('INFO')
quick_report = run_quick_pipeline(
    config_path=config_path,
    scenario='all',
    force=False,
)

print(f"Quick-run report: {quick_report['report_path']}")

2026-07-17 12:17:58 | INFO | Quick-run sample ready: 100000 train rows and 25000 test rows (reused).
2026-07-17 12:17:58 | INFO | Pipeline started for s1_none, s2_class_weight, s3_upsampling, s4_downsampling. Forced phases: none.
2026-07-17 12:17:58 | INFO | Phase 1/7 started: Dataset inspection (reused).
2026-07-17 12:17:58 | INFO | Phase 1/7 completed in 0.00 seconds.
2026-07-17 12:17:58 | INFO | Phase 2/7 started: Preprocessing (reused).
2026-07-17 12:17:58 | INFO | Phase 2/7 completed in 0.00 seconds.
2026-07-17 12:17:58 | INFO | Phase 3/7 started: Autoencoder training (reused).
2026-07-17 12:17:58 | INFO | Phase 3/7 completed in 0.00 seconds.
2026-07-17 12:17:58 | INFO | Phase 4/7 started: Latent feature extraction (reused).
2026-07-17 12:17:58 | INFO | Phase 4/7 completed in 0.00 seconds.
2026-07-17 12:17:58 | INFO | Phase 5/7 started: Class imbalance handling (validated).
2026-07-17 12:17:58 | INFO | Phase 5/7 completed in 0.09 seconds.
2026-07-17 12:17:58 | INFO | Phase 6/7 sta

Quick-run report: E:\Paper Multiclass-Intrusion-Detection-System\data\interim\quick_run\results\metrics\quick_run_report.json


In [4]:
# Cell 4 - Summarize sampling reduction and measured execution time.
performance_fields = [
    'train_rows',
    'test_rows',
    'train_row_reduction_percentage',
    'test_row_reduction_percentage',
    'sample_preparation_seconds',
    'pipeline_seconds',
    'total_seconds',
]
pd.Series({field: quick_report[field] for field in performance_fields}, name='value').to_frame()

,value
train_rows,100000.000000
test_rows,25000.000000
train_row_reduction_percentage,97.663806
test_row_reduction_percentage,97.663807
sample_preparation_seconds,0.056445
pipeline_seconds,4.380386
total_seconds,4.448948


In [5]:
# Cell 5 - Compare recorded cold-rebuild and cached quick-run timings.
quick_history = pd.read_csv(quick_report['history_path'])
quick_history.tail(10)

,completed_at,scenario,sample_regenerated,train_rows,test_rows,autoencoder_epochs,lightgbm_estimators,train_row_reduction_percentage,test_row_reduction_percentage,sample_preparation_seconds,pipeline_seconds,total_seconds,source_artifacts_unchanged
0,2026-07-17T11:11:18+07:00,all,True,100000,25000,5,50,97.663806,97.663807,15.254731,15.803466,31.119001,True
1,2026-07-17T11:11:40+07:00,all,False,100000,25000,5,50,97.663806,97.663807,0.031476,4.065478,4.105354,True
2,2026-07-17T11:13:51+07:00,all,True,100000,25000,5,50,97.663806,97.663807,0.343266,15.373538,15.777202,True
3,2026-07-17T11:14:10+07:00,all,False,100000,25000,5,50,97.663806,97.663807,0.032168,4.033645,4.074792,True
4,2026-07-17T11:14:34+07:00,s1,True,100000,25000,5,50,97.663806,97.663807,0.325007,6.653142,7.040083,True
5,2026-07-17T11:14:51+07:00,all,False,100000,25000,5,50,97.663806,97.663807,0.026126,11.964433,11.998715,True
6,2026-07-17T12:18:02+07:00,all,False,100000,25000,5,50,97.663806,97.663807,0.056445,4.380386,4.448948,True


In [6]:
# Cell 6 - Verify class coverage and inspect the sampled split distributions.
distribution_path = quick_root / 'results' / 'metrics' / 'split_class_distribution.csv'
quick_distribution = pd.read_csv(distribution_path)
assert quick_distribution.groupby('split')['class_index'].nunique().eq(10).all()
assert (quick_distribution['sample_count'] > 0).all()
quick_distribution

,split,class_index,class,source_count,sample_count,sample_percentage
0,train,0,Benign,2011247,46939,46.939
1,train,1,Backdoor,21716,459,0.459
2,train,2,DDoS,162,100,0.100
3,train,3,DoS,116,100,0.100
4,train,4,Injection,222157,5142,5.142
5,train,5,MITM,414,100,0.100
6,train,6,Password,272166,6310,6.310
7,train,7,Ransomware,4078,100,0.100
8,train,8,Scanning,28964,629,0.629
9,train,9,XSS,1719446,40121,40.121


In [7]:
# Cell 7 - Display status and duration for each phase in the quick pipeline.
with open(quick_report['pipeline_report_path'], 'r', encoding='utf-8') as file:
    pipeline_report = json.load(file)

phase_timing = pd.DataFrame(pipeline_report['phases'])
phase_timing

,phase,name,status,duration_seconds
0,1,Dataset inspection,reused,0.000001
1,2,Preprocessing,reused,0.000320
2,3,Autoencoder training,reused,0.001110
3,4,Latent feature extraction,reused,0.000674
4,5,Class imbalance handling,validated,0.090923
5,6,LightGBM training,validated,0.931762
6,7,Evaluation and analysis,executed,3.339706


In [8]:
# Cell 8 - Confirm isolation, source integrity, and smoke-test-only labeling.
full_processed_dir = resolve_project_path(config['paths']['processed_dir'], project_root)
quick_processed_dir = quick_root / 'processed'

assert quick_report['source_artifacts_unchanged'] is True
assert quick_report['scientific_use'] == 'smoke_test_only_not_final_results'
assert quick_report['train_row_reduction_percentage'] > 90.0
assert quick_report['test_row_reduction_percentage'] > 90.0
assert quick_processed_dir.resolve() != full_processed_dir.resolve()
assert phase_timing['phase'].tolist() == list(range(1, 8))

print('Quick-run isolation and integrity checks passed.')

Quick-run isolation and integrity checks passed.
